# 🎬 AutoCut Video AI — Server Komputasi Cloud Colab (V1.2.0 Protected)

> ⏰ **Rilis:** V1.2.0 (Server Rendering Video Edition)
> 🛡️ **Anti-Timeout Cloudflare:** Asynchronous Job Queue (`/api/v1/render` & `/api/v1/job/status`)
> ⚡ **Ultra-Fast RAM-Disk:** I/O 4.000 MB/s di `/dev/shm` (Bebas Bottleneck SSD)
> 🚀 **Zero Pop-Up Google Drive:** Server berjalan instan tanpa izin akses Drive (100% aman).
> 🔐 **Protected Core Engine:** Modul terenkripsi resmi PyArmor Linux x86_64.

Notebook ini berfungsi sebagai **Engine Backend Rendering Video Klip** untuk aplikasi **Intisari AutoCut Android**.

### ✨ Fitur Unggulan V1.2.0:
1. 📱 **Scan-to-Pairing (QR Code)**: URL tunnel Cloudflare otomatis dikonversi ke gambar QR Code di layar Colab untuk pairing kamera 1-detik dari smartphone.
2. ⚡ **Gigabit YouTube Downloader**: Mengunduh video master YouTube dalam hitungan 1–2 detik.
3. 🎙️ **Sistem Subtitel Hibrida**: Transkripsi lokal otomatis menggunakan `faster-whisper` (100% gratis tanpa API key) + fallback opsi Groq API.
4. 🎬 **CapCut-Certified 9:16 Reframe**: Format YUV420p, CFR 30 FPS, GOP 60, AAC stereo studio, dan blur background sinematik.
5. ⏱️ **Watchdog Auto-Shutdown**: Otomatis mematikan runtime Colab jika tidak ada aktivitas render baru selama 10 menit (hemat sesi Colab).

---
### 🚀 Cara Menjalankan:
1. Klik tombol **Play (▶)** di sel kode di bawah ini.
2. Tunggu hingga proses setup selesai dan **Tautan Tunnel** serta **QR Code** muncul di layar.
3. Buka aplikasi **Intisari AutoCut Android** di HP Anda -> Buka Tab **Pengaturan** -> Pindai QR Code atau salin tautan tersebut.


In [ ]:
"""
🎬 AUTOCUT VIDEO ENGINE — SERVER RENDERING & CLOUDFLARE TUNNEL (V1.2.0)
Hak Cipta (C) 2026 IntisariApps.com. Seluruh hak cipta dilindungi.
"""

# @title ⚙️ PUSAT KENDALI ENGINE RENDERING COLAB
# @markdown Atur parameter sesi Colab di bawah ini:
AUTO_SHUTDOWN_MINUTES = 10  # @param [0, 5, 10, 15, 30] {type:"raw"}
GROQ_API_KEY = ""  # @param {type:"string"}

import os
import sys
import time
import re
import shutil
import zipfile
import subprocess
import sysconfig
import threading

print("=" * 80)
print("🚀 MEMULAI AUTOCUT VIDEO ENGINE V1.2.0 (BYOC SERVER)")
print("=" * 80)

# 1. Download binary cloudflared jika belum ada
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⏳ [1/4] Mengunduh Cloudflare Tunnel client...")
    subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
print("✅ Cloudflare Tunnel siap!")

# 2. Install dependensi sistem dasar & fast libraries
print("📦 [2/4] Memeriksa dependensi sistem (FastAPI, yt-dlp, faster-whisper, qrcode)...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "fastapi", "uvicorn[standard]", "python-multipart", "yt-dlp", "faster-whisper", "requests", "qrcode[pil]", "pydantic"
], check=False)

# 3. Unduh modul engine biner resmi terenkripsi (PyArmor Linux x86_64)
print("🔐 [3/4] Mengunduh modul biner terenkripsi: autocut_video_engine.zip...")
pkg_url = f"https://raw.githubusercontent.com/intisariapps-com/intisariAutoCutPublic/main/autocut_video_engine.zip?t={int(time.time())}"
pkg_local = "/tmp/autocut_video_engine.zip"
subprocess.run(["wget", "-q", "--no-cache", "--no-cookies", "-O", pkg_local, pkg_url], check=True)

site_pkg = sysconfig.get_paths()["purelib"]
with zipfile.ZipFile(pkg_local, "r") as zf:
    zf.extractall(site_pkg)

# Bersihkan cache modul Python agar selalu memuat biner terbaru
for mod in list(sys.modules.keys()):
    if "autocut_video_engine" in mod or "pyarmor" in mod:
        del sys.modules[mod]

import autocut_video_engine
print("✅ Modul Engine AutoCut berhasil dipasang di memori!")

# 4. Setel environment & Watchdog
if GROQ_API_KEY.strip():
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY.strip()

# Jalankan server FastAPI di background thread pada port 8000
print("🌐 [4/4] Meluncurkan server backend pada port 8000...")
import uvicorn
from autocut_video_engine.server import app as fastapi_app

def start_uvicorn():
    uvicorn.run(fastapi_app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)

# 5. Jalankan Cloudflare Tunnel dan ambil URL publik
print("🚇 Membuka Cloudflare Quick Tunnel...")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    universal_newlines=True
)

public_tunnel_url = None
timeout_sec = 25
start_t = time.time()
while time.time() - start_t < timeout_sec:
    line = tunnel_proc.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_tunnel_url = match.group(0)
            break
    time.sleep(0.1)

if not public_tunnel_url:
    print("⚠️ Gagal mendapatkan URL Cloudflare otomatis. Periksa log koneksi.")
else:
    print("\n" + "=" * 80)
    print(f"🎉 SERVER BERHASIL ONLINE!")
    print(f"👉 URL Tunnel Publik: {public_tunnel_url}")
    print("=" * 80)

    # Tampilkan QR Code di layar Colab untuk pairing kamera smartphone
    try:
        import qrcode
        from IPython.display import display, Image
        import io
        qr = qrcode.QRCode(box_size=8, border=2)
        qr.add_data(public_tunnel_url)
        qr.make(fit=True)
        img = qr.make_image(fill_color="black", back_color="white")
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        print("\n📱 PINDAI QR CODE DI BAWAH INI DARI APLIKASI ANDROID:")
        display(Image(buf.getvalue()))
    except Exception as e_qr:
        print(f"(QR Code viewer fallback: {e_qr})")

    print("\nℹ️ Server siap menerima instruksi render video dari aplikasi Android!")
    print("⏱️ Tekan tombol Stop (⏹) kapan saja untuk mematikan server.")

    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\n🛑 Sesi server dihentikan.")
